# House Prices — XGBoost và Lasso 80/20

Kết hợp dự đoán ln(SalePrice) đã lưu trên cùng 5 fold. XGBoost dùng 79 cột + TotalSF + TotalBath. Lasso thêm cả ba cột tổng hợp. Hai mô hình nguồn chỉ lọc outlier trong phần huấn luyện. Tỷ lệ cố định 80/20. Kết quả mang tính thăm dò vì các cấu hình nguồn được chọn trên cùng dữ liệu.

In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
project = next(p for p in (cwd, *cwd.parents, cwd / "Challenge1_House Prices_ Advanced Regression")
               if (p / "src" / "prepare_data.py").is_file())
experiments = project / "experiments"
xgb_dir = experiments / "15-11-2026_XGBoost_5bo_outlier_phuc" / "outputs" / "20261115_104726_284284"
lasso_dir = experiments / "15-11-2026_5model_5bo_outlier_phuc" / "outputs" / "20261115_110212_593276"
keys = ["Id", "fold"]
target = "ln_SalePrice"
xgb_col = "xgboost_TotalSF + TotalBath"
lasso_col = "Lasso | 82_ca_ba | filter_train"
xgb = pd.read_csv(xgb_dir / "validation_predictions.csv", usecols=[*keys, target, xgb_col])
lasso = pd.read_csv(lasso_dir / "validation_predictions.csv", usecols=[*keys, target, lasso_col])
pred = xgb.merge(lasso, on=keys, validate="one_to_one", suffixes=("", "_lasso"))
assert len(pred) == len(xgb) == len(lasso) == 1460 and pred["Id"].is_unique
assert pred.groupby("fold").size().to_dict() == dict.fromkeys(range(1, 6), 292)
np.testing.assert_allclose(pred[target], pred.pop(target + "_lasso"), rtol=0, atol=1e-12)
pred = pred.rename(columns={xgb_col: "XGBoost", lasso_col: "Lasso"})

In [2]:
pred["Ensemble 80/20"] = 0.8 * pred["XGBoost"] + 0.2 * pred["Lasso"]
models = ["XGBoost", "Lasso", "Ensemble 80/20"]
assert np.isfinite(pred[models + [target]]).all().all()
errors = pred[models].subtract(pred[target], axis=0)
scores = errors.pow(2).groupby(pred["fold"]).mean().pow(0.5)
cols = [f"fold{i}" for i in scores.index]
xgb_metrics = pd.read_csv(xgb_dir / "validation_metrics.csv").set_index("variant")
lasso_metrics = pd.read_csv(lasso_dir / "validation_metrics.csv").set_index(["model", "variant", "policy"])
# CSV float32 prediction rounding can change recomputed RMSE by a few 1e-8.
np.testing.assert_allclose(scores["XGBoost"], xgb_metrics.loc["TotalSF + TotalBath", cols].astype(float), atol=1e-7)
np.testing.assert_allclose(scores["Lasso"], lasso_metrics.loc[("Lasso", "82_ca_ba", "filter_train"), cols].astype(float), atol=1e-7)
summary = pd.DataFrame({"rmse_log_mean": scores.mean(), "rmse_log_std": scores.std(ddof=1)})
summary["delta_vs_xgboost"] = summary["rmse_log_mean"] - summary.loc["XGBoost", "rmse_log_mean"]
summary["folds_improved"] = scores.lt(scores["XGBoost"], axis=0).sum()
print(scores.round(5).to_string())
print(summary.round(5).to_string())

      XGBoost    Lasso  Ensemble 80/20
fold                                  
1     0.13710  0.13009         0.13311
2     0.11221  0.10897         0.10760
3     0.16284  0.22871         0.17122
4     0.12474  0.11512         0.11854
5     0.11572  0.10326         0.10919
                rmse_log_mean  rmse_log_std  delta_vs_xgboost  folds_improved
XGBoost               0.13052       0.02047           0.00000               0
Lasso                 0.13723       0.05211           0.00670               4
Ensemble 80/20        0.12793       0.02624          -0.00259               4


In [3]:
stamp = datetime.now(timezone(timedelta(hours=7))).strftime("%Y%m%d_%H%M%S_%f")
run = experiments / "15-11-2026_ensemble_xgboost_lasso_phuc" / "outputs" / stamp
run.mkdir(parents=True, exist_ok=False)
pred.to_csv(run / "validation_predictions.csv", index=False)
scores.to_csv(run / "validation_by_fold.csv")
summary.to_csv(run / "validation_metrics.csv", index_label="model")
print(run.relative_to(project))

experiments\15-11-2026_ensemble_xgboost_lasso_phuc\outputs\20261115_111145_484418
